# Co-location check: chunk tile vs the full field

1. **Load a co-location result** — the final parquet if it exists, else
   assembled from the per-property checkpoint fragments.
2. **Compute both sides here** — the tile from its CHUNKS store, and the global
   reference for the same snapshot — so it does not matter how far a production
   run has got.
3. **Compare** for the fronts they share.

The checkpoint fragments hold *only* their own property's columns — no `flabel`,
no `npix` — because they are written positionally alongside the run's label
array. The key is rebuilt below from the label map; a fragment whose row count
disagrees with the front count came from a run with a different `min_npix` or
label map, and the assert catches it.

A surviving `colocate_ckpt_*` next to a finished parquet is litter from
`shutil.rmtree` failing on a network filesystem. Delete it before re-running
that timestamp with clobber.

In [ ]:
import glob, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT     = '/mnt/tank/Oceanography/data/OGCM/LLC/Fronts'
FRONTS   = f'{ROOT}/V5/SURF'
RUN_ID   = 'v2_2_01'                  # the build_v5 run that made the fronts
FIND_CFG = 'D'
RUN_TAG  = f'{RUN_ID}_bfronts'        # how the products are named

TS_DIR   = f'{FRONTS}/20111204_000000'

print(sorted(os.listdir(TS_DIR)))
ck = f'{TS_DIR}/colocate_ckpt_{RUN_TAG}'
if os.path.isdir(ck):
    print(f'\n{len(os.listdir(ck))} checkpoint fragments: {sorted(os.listdir(ck))}')

## 1. Load one timestamp's co-location

In [ ]:
def load_colocation(ts_dir, run_tag, min_npix=1):
    """Final parquet if present, else assemble the checkpoint fragments.

    Returns (df, source).  Fragments carry only their own property columns, so
    flabel/npix are recomputed from the label map and concatenated on.  The
    label map is looked up one level up when *ts_dir* is a subdirectory (a tile
    or nb_check output).
    """
    final = glob.glob(f'{ts_dir}/front_properties_*_{run_tag}.parquet')
    if final:
        return pd.read_parquet(final[0]), 'final parquet'

    frags = sorted(glob.glob(f'{ts_dir}/colocate_ckpt_{run_tag}/*.parquet'))
    if not frags:
        raise FileNotFoundError(f'no result and no fragments in {ts_dir}')

    lab = (glob.glob(f'{ts_dir}/labeled_fronts_global_*_{run_tag}.npy')
           or glob.glob(f'{ts_dir}/../labeled_fronts_global_*_{run_tag}.npy'))
    labels, counts = np.unique(np.load(lab[0]), return_counts=True)
    keep = (labels > 0) & (counts >= min_npix)
    key = pd.DataFrame({'flabel': labels[keep], 'npix': counts[keep]})

    parts = [key]
    for f in frags:
        d = pd.read_parquet(f)
        assert len(d) == len(key), (
            f'{os.path.basename(f)}: {len(d)} rows vs {len(key)} fronts -- '
            'fragment came from a run with a different label map or min_npix')
        parts.append(d)
    return pd.concat(parts, axis=1), f'{len(frags)} checkpoint fragments'


df, src = load_colocation(TS_DIR, RUN_TAG)
print(f'{len(df):,} fronts from {src},  {len(df.columns)} columns')
df.head()

## 2. Sanity: how much is NaN, and why

In [ ]:
means = df.filter(like='_mean')
nan_pct = (means.isna().mean() * 100).sort_values(ascending=False)
print('% of fronts with no valid pixels, per property:')
print(nan_pct.round(2).to_string())

print(f'\nnpix: min={df.npix.min()}  median={df.npix.median():.0f}  '
      f'max={df.npix.max():,}  total front pixels={df.npix.sum():,}')
print(f'single-pixel fronts: {(df.npix == 1).sum():,} '
      f'({100 * (df.npix == 1).mean():.1f}%)  <- these drive the '
      '"Degrees of freedom <= 0" warnings')

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].hist(np.log10(df.npix), bins=60)
ax[0].set(xlabel='log10(npix)', ylabel='fronts', title='front size')
ax[1].barh(nan_pct.index, nan_pct.values)
ax[1].set(xlabel='% NaN', title='fronts with no valid pixels')
fig.tight_layout()

A high NaN rate on `SIarea` is the ice mask working. A high rate on a field that
should be everywhere (`density`, `gradb2`) means land, or something wrong.

## 3. Compute both sides here

`2012-07-04 12:00:00` is the only date the 100-timestep global run and the
Monterey chunk store share, and `density` / `gradb2` are the only two properties
in both — so that snapshot and those two fields are the whole comparison.

Both sides are computed here, so it does not matter how far the production run
has got. The global side writes to a `nb_check/` subdirectory with its own
checkpoint cache, so it cannot collide with a run in flight. Neither side is
ice-masked, which matches the chunk path exactly (and Monterey has no ice).

First run costs: the rect↔face lookup maps get built (three 13x4320x4320 index
arrays, tens of seconds and a few GB, then cached for the session), and each
property is read or computed from S3. `clobber=False`, so re-running is cheap.

In [ ]:
from fronts.llc import io as llc_io
from fronts.llc import tiles as llc_tiles
from fronts.properties.run import colocate_fronts, colocate_tile

CHUNK      = 'monterey_bay'
TS         = '2012-07-04T12_00_00'
CMP_DIR    = f'{FRONTS}/20120704_120000'
SHARED     = ['density', 'gradb2']
GLOBAL_OUT = f'{CMP_DIR}/nb_check'
CONFIG     = '../../runs/prototypes/one_full/run_v5_100_timesteps.yaml'

llc_io.set_fronts_path(ROOT)
llc_io.set_run_layout('V5/SURF', file_tag=RUN_ID)

# --- tile side: fields computed from the chunk store ---
tile = llc_tiles.tile_from_chunk_store(CHUNK)
print(f'{CHUNK}: tile {tile.tile_idx}, face {tile.face_idx}, '
      f'rect j={tile.rect_j_slice.start} i={tile.rect_i_slice.start}, '
      f'face-local j={tile.j_face_slice.start} i={tile.i_face_slice.start}\n')

colocate_tile(TS, FIND_CFG, RUN_ID,
              property_names=SHARED, tile=tile,
              percentiles=[90], clobber=False,
              loader=llc_tiles.chunk_loader(CHUNK, TS))

In [ ]:
# --- global side: same two fields, straight from the global zarr stores ---
colocate_fronts(TS, FIND_CFG, RUN_ID,
                property_names=SHARED,
                output_dir=GLOBAL_OUT,
                percentiles=[90], skip_missing=True, clobber=False,
                config_file=CONFIG, source='zarr', ice_mask=False)

## 4. Compare

Fronts wholly inside the tile (`npix_tile == npix_glob`) are the fair
comparison — a clipped front's tile statistics describe only its clipped part.

Expect close but not identical: the global fields come from the full face with
grid connections, the tile fields from a single-face chunk with
`use_connections=False`, so stencil-based quantities differ near the rim.

In [ ]:
tile_dir = f'{CMP_DIR}/tile{tile.tile_idx:03d}'

g, gsrc = load_colocation(GLOBAL_OUT, RUN_TAG)
t, tsrc = load_colocation(tile_dir, RUN_TAG)
print(f'global: {len(g):,} fronts ({gsrc})')
print(f'tile  : {len(t):,} fronts ({tsrc})')

m = g.merge(t, on='flabel', suffixes=('_glob', '_tile'))
inside = m[m.npix_tile == m.npix_glob]
print(f'\nfronts in both: {len(m):,}   wholly inside the tile: {len(inside):,} '
      f'({100 * len(inside) / max(len(m), 1):.0f}%)')

In [ ]:
if not len(inside):
    print('every shared front is clipped by the tile edge; nothing fair to compare')
else:
    fig, axes = plt.subplots(1, len(SHARED), figsize=(5 * len(SHARED), 4.5))
    axes = np.atleast_1d(axes)

    for ax, prop in zip(axes, SHARED):
        a = inside[f'{prop}_mean_glob'].values
        b = inside[f'{prop}_mean_tile'].values
        ok = np.isfinite(a) & np.isfinite(b)
        a, b = a[ok], b[ok]
        if not len(a):
            ax.set_title(f'{prop}: no finite pairs')
            continue

        ax.scatter(a, b, s=6, alpha=0.3)
        lo, hi = np.nanpercentile(np.r_[a, b], [0.5, 99.5])
        ax.plot([lo, hi], [lo, hi], 'k--', lw=1)
        ax.set(xlabel=f'{prop} global', ylabel=f'{prop} tile',
               title=f'{prop}  (n={len(a):,})')

        denom = np.where(np.abs(a) > 0, np.abs(a), np.nan)
        rel = np.abs(b - a) / denom
        print(f'{prop:12} median |rel diff| = {np.nanmedian(rel):.2e}   '
              f'p95 = {np.nanpercentile(rel, 95):.2e}   '
              f'max = {np.nanmax(rel):.2e}')

    fig.tight_layout()

Median relative difference near machine precision means the two paths agree and
the tile orientation is right. A systematic offset, or a scatter splitting into
two branches, points at a mis-paired label map — check `labels_for_tile` against
the tile's `XC`/`YC` if that shows up.